In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

import numpy as np
import scipy.stats as stats

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
url = "winequality-white.csv"
data = pd.read_csv(url, sep=";")

In [36]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [37]:
cols2zscore = data.keys()
cols2zscore = cols2zscore.drop('quality')

In [38]:
def createAdataset(qualThreshold):

    data['boolQuality'] = 0

    tmp = ( data['quality']>qualThreshold ).astype(int)
    data['boolQuality'] = tmp

    dataT = torch.tensor( data[cols2zscore].values ).float()
    labels = torch.tensor( data['boolQuality'].values ).float()
    labels = labels[:,None]

    train_data, test_data, train_labels, test_labels = train_test_split(dataT, labels, test_size=0.2)

    train_data = TensorDataset(train_data, train_labels)
    test_data = TensorDataset(test_data, test_labels)

    batchsize = 8
    train_loader = DataLoader(train_data, batch_size=batchsize, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_data, batch_size=test_data.tensors[0].shape[0])

    return train_loader, test_loader

In [39]:
train_loader, test_loader = createAdataset(4)

qualityRatings = train_loader.dataset.tensors[1].detach()
print('%s low-rated wines' %torch.sum(qualityRatings==0).item())
print('%s igh-rated wines' %torch.sum(qualityRatings==1).item())

151 low-rated wines
3767 igh-rated wines


In [40]:
class ANNwine(nn.Module):
    def __init__(self):
        super().__init__()

        self.input = nn.Linear(11,16)
        
        self.fc1 = nn.Linear(16,32)
        self.fc2 = nn.Linear(32,32)

        self.output = nn.Linear(32,1)

    def forward(self,x):
        x = F.leaky_relu( self.input(x) )
        x = F.leaky_relu( self.fc1(x) )
        x = F.leaky_relu( self.fc2(x) )
        return self.output(x)

In [41]:
numepochs = 500

def trainTheModel():

    lossfun = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(winenet.parameters(), lr=.001)

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    for epochi in range(numepochs):

        batchAcc = []
        batchLoss = []
        for X,y in train_loader:

            yHat = winenet(X)
            loss = lossfun(yHat, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batchLoss.append(loss.item())

            batchAcc.append( 100*torch.mean(((yHat>0) == y).float()).item() )

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        X,y = next(iter(test_loader))
        with torch.no_grad():
            yHat = winenet(X)
        testAcc.append(100*torch.mean(((yHat>0) == y).float()).item() )

    return trainAcc, testAcc, losses

In [42]:
train_loader, test_loader = createAdataset(5)
winenet = ANNwine()
trainAcc, testAcc, losses = trainTheModel()

X,y = next(iter(test_loader))
yHat = winenet(X)
itemAccuracy = ((yHat>0) == y).float()

perQualAcc = [ 100*torch.mean(itemAccuracy[y==0]),
              100*torch.mean(itemAccuracy[y==1]) ]
perQualAcc

[tensor(70.9677), tensor(75.0746)]

In [ ]:
fig, ax = plt.subplots(3,3,figsize=(18,12))

qualityThresholds = [4,5,6]

for quali in range(len(qualityThresholds)):
    train_loader, test_loader = createAdataset( qualityThresholds[quali] )
    winenet = ANNwine()
    trainAcc, testAcc, losses = trainTheModel()

    X,y = next(iter(test_loader))
    itemAccuracy = ((winenet(X)>0) == y).float()
    perQualAcc = [ 100*torch.mean(itemAccuracy[y==0]), 
                100*torch.mean(itemAccuraccy[y==1]) ]

    ax[quali,0].plot(losses)
    ax[quali,0].set_title('Losses with qualthresh ' + str(qualityThresholds[quali]+.5))
    ax[quali,0].set_xlabel('Epoch')
    ax[quali,0].grid()

    ax[quali,1].plot(trainAcc, label='Train')
    ax[quali,1].plot(testAcc, label='Test')
    ax[quali,1].set_title('Accuracy with qualthresh ' + str(qualityThresholds[quali]+.5))
    ax[quali,1].legend()
    ax[quali,1].set_xlabel('Epoch')
    ax[quali,1].set_ylim([0,100])
    ax[quali,1].grid()

    bh = ax[quali,2].bat(['Bad', 'Good'], perQualAcc)
    ax[quali,2].set_ylim([0,100])
    ax[quali,2].set_xlabel('Wine quality')
    ax[quali,2].set_ylabel('Test accuracy')
    ax[quali,2].set_title('Per-qual acc. with qualthresh ' + str(qualityThresholds[quali]+.5))

    for i,r in enumerate(bh):
        N = torch.sum(train_loader.dataset.tensors[1]==i).item()
        ax[quali,2].text(r.get_x()+r.get_width()/2, r.get_height()+1,
                    'N=%s'%N, ha='center', va='bottom', fontsize=14)

plt.tight_layout
plt.show()